In [3]:
# Load Dependencies
import numpy as np
import pandas as pd
from itertools import product
from scipy.interpolate import interp1d
import itertools
from itertools import product
import sys
import re
import statistics
#sys.path.append('/Users/meoskin/Jupyter')

BASE_THRESHOLD = 0.95
HIATUS_THRESHOLD = 0.975
CORRELATION_THRESHOLD = 0.05
MINIMUM_EVENT_QUALITY = 1
MINIMUM_JOINT_PROBABILITY = 1e-10

In [1061]:
# Define Earthquake Event class [and associated helper functions]
class Earthquake:

    #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
    def __init__(self, name, age, PDF, CDF):
        self.name = name
        self.age = np.array(age)
        self.PDF = np.array(PDF)
        self.CDF = np.array(CDF)
    #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
    def event(self):
        r = np.random.rand()
        return self.age[np.argmin(np.abs(self.CDF - r))]
    #Returns n sampled ages: calls event() n times, then stores results in an array
    def events(self, n):
        return np.array([self.event() for _ in range(n)])
    #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
    def resample(self):
        L = int(self.age[-1] - self.age[0])
        new_age = np.linspace(self.age[0], self.age[-1], L + 1)

        f_age = interp1d(self.age, self.age, fill_value="extrapolate")
        f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
        f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

        self.age = f_age(new_age)
        self.PDF = f_pdf(new_age)
        self.PDF = self.PDF / np.sum(self.PDF)
        self.CDF = f_cdf(new_age)

    
    def BC(self,other):
        # Make sure both objects have the same sampling interval
        diff_age = np.mean(np.diff(self.age))
        if diff_age != np.mean(np.diff(other.age)):
            raise ValueError("Sampling interval differs")

        # Make sure both objects sample the same times of the year
        misalign = self.age[0] % 1 - other.age[0] % 1
        if not np.isclose(misalign,0): # if the array of ages are on different times of the year, shift so they allign so that BC can be calculated (only basal boundary ages fall on whole-number years)
        # shift the two distributions so they line up
            self_age = self.age.copy()
            self_age = self_age + misalign
            self_PDF = np.interp(self_age, self.age, self.PDF)
            # calculate Bhattacharya Coefficient
            this_df = pd.DataFrame({'age':self_age, 'this_pdf':self_PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['root_product'] = np.sqrt(overlap_df['this_pdf'] * overlap_df['that_pdf'])
            return(sum(overlap_df['root_product']))
            
        else: #If distributions are not misaligned
            # calculate Bhattacharya Coefficient
            this_df = pd.DataFrame({'age':self.age, 'this_pdf':self.PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['root_product'] = np.sqrt(overlap_df['this_pdf'] * overlap_df['that_pdf'])
            return(sum(overlap_df['root_product']))
                             
    
    #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
    def jointprobability(self, other,name='Joint'):
        # Make sure both objects have the same sampling interval
        diff_age = np.mean(np.diff(self.age))
        if diff_age != np.mean(np.diff(other.age)):
            raise ValueError("Sampling interval differs")

        # Make sure both objects sample the same times of the year
        misalign = self.age[0] % 1 - other.age[0] % 1
        if not np.isclose(misalign,0): # if the array of ages are on different times of the year, shift so they allign so that BC can be calculated (only basal boundary ages fall on whole-number years)
        # shift the two distributions so they line up
            self_age = self.age.copy()
            self_age = self_age + misalign
            self_PDF = np.interp(self_age, self.age, self.PDF)
            # calculate joint probability
            this_df = pd.DataFrame({'age':self_age, 'this_pdf':self_PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            if this_df.empty or that_df.empty: raise ValueError('Warning: One or more dataframes empty, check age allignment!')
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['joint_pdf'] = overlap_df['this_pdf'] * overlap_df['that_pdf']
            total_joint_probability = sum(overlap_df['joint_pdf'])
            if total_joint_probability < MINIMUM_JOINT_PROBABILITY: raise RuntimeError('Sum joint probability close to zero!')
            overlap_df['joint_pdf'] = overlap_df['joint_pdf']/total_joint_probability
            overlap_df['joint_cdf'] = np.cumsum(overlap_df['joint_pdf'])
            return Earthquake(name, overlap_df['age'],overlap_df['joint_pdf'],overlap_df['joint_cdf'])
            
        else: #If distributions are not misaligned
            # calculate joint probability
            this_df = pd.DataFrame({'age':self.age, 'this_pdf':self.PDF})
            that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
            overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
            overlap_df['joint_pdf'] = overlap_df['this_pdf'] * overlap_df['that_pdf']
            total_joint_probability = sum(overlap_df['joint_pdf'])
            if total_joint_probability < MINIMUM_JOINT_PROBABILITY: raise RuntimeError('Sum joint probability close to zero!')
            overlap_df['joint_pdf'] = overlap_df['joint_pdf']/total_joint_probability
            overlap_df['joint_cdf'] = np.cumsum(overlap_df['joint_pdf'])
            return Earthquake(name, overlap_df['age'],overlap_df['joint_pdf'],overlap_df['joint_cdf'])


class Candidate_Event(Earthquake):
    def __init__(self, name, event_list=[]):
        self.name = name
        self.event_list = event_list
        joint = event_list[next((i for i, x in enumerate(event_list) if x.name.split('_', 1)[-1] != 'basal_boundary'), None)] # find the index of the first non-basal boundary object in the candidate
        for event in event_list[1:len(event_list)]:
            if event.name.split('_', 1)[-1] == 'basal_boundary': # don't calculate joint probability with basal boundaries
                continue
            else:
                joint = joint.jointprobability(event)
        self.age = list(joint.age)
        self.PDF = list(joint.PDF)
        self.CDF = list(joint.CDF)

In [733]:
# Define paleoseismic dataset object
class paleoseismic_dataset:
    # constructor stores essential site, fault, and position information
    def __init__(self,site_name,fault_name,position,slip_threshold = 0):
        self.site_name = site_name
        self.fault_name = fault_name
        self.position = position
        self.slip_threshold = slip_threshold
        self.exclude = []
        self.event_names = []

    def load_oxcal(self,filename, name_selector='Event', base_selector='Base', 
                   hiatus_selector = 'Hiatus', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]

        # Subset for events and extract

        df_events = df[df["name"].str.contains(name_selector, na=False)]

        self.events = []

        for eventname, group in df_events.groupby("name", sort=False): 
            eventname_fixed = re.sub(r"\s+", "_", eventname)
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]
    
            self.events.append(
                Earthquake(
                    name=self.site_name + "_" + eventname_fixed,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )

        # Extract event names and produce generic event quality tuple:
        for event in self.events:
            self.event_names.append(event.name)
        self.event_quality = dict(zip(self.event_names, [4] * len(self.events))) # set default event quality to 4

        # Subset for basal boundary
        df_base = df[df["name"].str.contains(base_selector, na=False)]
        ages = df_base["value"].values
        pdf = df_base["probability"].values
        cdf = np.cumsum(pdf)
        self.basal_boundary = ages[np.searchsorted(cdf, 1 - BASE_THRESHOLD)]
##ERIK EXPERIMENTING WITH STUFF
        self.base = []
        self.base = Earthquake(name=self.site_name+"_basal_boundary",age=ages,PDF=pdf,CDF=cdf)
##

        # Subset hiatuses
        df_hiatus = df[df["name"].str.contains(hiatus_selector, na=False)]
        self.hiatuses = []

        hiatusnumber = 1
        for hiatus, group in df_hiatus.groupby("name", sort=False):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            self.hiatuses.append(
                Earthquake(
                    name=self.site_name + "_hiatus_"+str(hiatusnumber),
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
            hiatusnumber = hiatusnumber + 1
                
    def get_marginal_pdfs(self,minimum_event_quality = 1):
        rows = []
        for i, eq in enumerate(self.events):
            if eq.name in self.exclude:
                continue
            if self.event_quality[eq.name] < minimum_event_quality:
                continue
            df = pd.DataFrame({
                "event": self.site_name + "_" + eq.name,
                "year": eq.age,
                "prob": eq.PDF,
                "CDF": eq.CDF
                
            })
            #df = df[df["year"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    def event_list(self):
        output_list = []
        for i, eq in enumerate(self.events):
            if eq.name in self.exclude:
                continue
            if self.event_quality[eq.name] < MINIMUM_EVENT_QUALITY:
                continue
            output_list.append(eq)
        return(output_list)

In [539]:
# Annette Paleoseismic Site

Annette_location = 20 # km
Annette = paleoseismic_dataset("Annette","San Andreas", Annette_location)
Annette.load_oxcal("C:/Users/hiker/Thesis/Annette_inverted.csv")
Annette.exclude = ['Annette_Groundshaking_Event_']
Annette.event_quality.update(Annette_Event_2 = 3, Annette_Event_3 = 4, Annette_Event_4 = 5, Annette_Event_5 = 4, Annette_Event_6 = 5)

In [399]:
# Bidart Paleoseismic Site

Bidart_location = 72 # km
Bidart = paleoseismic_dataset("Bidart", "San Andreas", Bidart_location)
Bidart.load_oxcal("C:/Users/hiker/Thesis/Bidart_Akciz2023_edited1.csv")
Bidart.exclude = ['Bidart_Event_1']
Bidart.event_quality.update(Bidart_Event_2 = 5, Bidart_Event_3 = 5, Bidart_Event_4 = 5, Bidart_Event_5 = 5, Bidart_Event_6 = 5)

In [401]:
# Frazier Mountain Paleoseismic Site
Frazier_location = 169 # km
Frazier = paleoseismic_dataset("Frazier","San Andreas", Frazier_location)
Frazier.load_oxcal("C:/Users/hiker/Thesis/FrazierMountain_HiatusesIncluded.csv")
Frazier.exclude = ['Frazier_Event_Historic_1857', 'Frazier_=Event_11'] #This weird 'extra' event comes from the new way we model hiatuses at the end of the OxCal script with a Sum()
#Frazier.event_quality.update(Frazier
Frazier.event_quality.update(Frazier_Event_2 = 5, Frazier_Event_3 = 5, Frazier_Event_4 = 5, Frazier_Event_5 = 5, Frazier_Event_6 = 3, Frazier_Event_7 = 5, Frazier_Event_8 = 4, Frazier_Event_9 = 5, Frazier_Event_10 = 3, Frazier_Event_11 = 4)
#print(Frazier.get_event_list())
#Frazier.get_marginal_pdfs(excluded_events)
#Frazier.event_quality
#Frazier.hiatuses[0].age#[np.where(Frazier.hiatuses[0].CDF > 0.1)[0]]

In [403]:
#Elizabeth Lake Paleoseismic Site

Elizabeth_location = 218 # km
Elizabeth = paleoseismic_dataset("Elizabeth", "San Andreas", Elizabeth_location)
Elizabeth.load_oxcal("C:/Users/hiker/Thesis/Elizabeth_HiatusesIncluded.csv")
Elizabeth.event_quality.update(Elizabeth_Event_EL2 = 3, Elizabeth_Event_EL3 = 3, Elizabeth_Event_EL4 = 3, Elizabeth_Event_ELX = 2)
#Elizabeth.hiatuses[1].age[np.where(Elizabeth.hiatuses[0].CDF > 0.1)[1]]
#Elizabeth.hiatuses[1].age
#Elizabeth.hiatuses[1].age

In [775]:
# Pallett Creek Paleoseismic Site

Pallett_location = 270 # km
Pallett = paleoseismic_dataset("Pallett","San Andreas", Pallett_location)
Pallett.load_oxcal("C:/Users/hiker/Thesis/PallettCreek_HiatusesIncluded.csv")
#print(Pallett.event_names)
Pallett.exclude = ['Pallett_Event_X', 'Pallett_Event_MRE']
Pallett.event_quality.update(Pallett_Event_V = 5, Pallett_Event_T = 5, Pallett_Event_R = 5, Pallett_Event_N = 4, Pallett_Event_I = 4, Pallett_Event_F = 5, Pallett_Event_D = 4, Pallett_Event_C = 4)
#Pallett.event_quality
#Pallett.get_marginal_pdfs(4)
#Pallett.hiatuses[0].age

In [406]:
#Wrightwood Paleoseismic Site

Wrightwood_location = 292 # km
Wrightwood = paleoseismic_dataset("Wrightwood", "San Andreas", Wrightwood_location)
Wrightwood.load_oxcal("C:/Users/hiker/Thesis/Wrightwood_HiatusesIncluded.csv")
print(Wrightwood.event_names)
Wrightwood.exclude = ['Wrightwood_Event_Historic_1812', 'Wrightwood_Event_Historic_1857']
Wrightwood.event_quality.update(Wrightwood_Event_3 = 5, Wrightwood_Event_4 = 5, Wrightwood_Event_5 = 5, Wrightwood_Event_6 = 5, Wrightwood_Event_7 = 4, Wrightwood_Event_8 = 4, Wrightwood_Event_9 = 4, Wrightwood_Event_10 = 4, Wrightwood_Event_11 = 5, Wrightwood_Event_12 = 4, Wrightwood_Event_13 = 3, Wrightwood_Event_14 = 4)
#Wrightwood.hiatuses[0].age[np.where(Wrightwood.hiatuses[0].CDF < 0.95)[0]]
#Wrightwood.hiatuses[1].age[np.where(Wrightwood.hiatuses[1].CDF < 0.95)[0]]

['Wrightwood_Event_W14', 'Wrightwood_Event_W13', 'Wrightwood_Event_W12', 'Wrightwood_Event_W11', 'Wrightwood_Event_W10', 'Wrightwood_Event_W9', 'Wrightwood_Event_W8', 'Wrightwood_Event_W7', 'Wrightwood_Event_W6', 'Wrightwood_Event_W5', 'Wrightwood_Event_W4', 'Wrightwood_Event_W3', 'Wrightwood_Event_Historic_1812', 'Wrightwood_Event_Historic_1857']


In [409]:
#Pitman Canyon Paleoseismic Site

Pitman_location = 317 # km
Pitman = paleoseismic_dataset("Pitman", "San Andreas", Pitman_location)
Pitman.load_oxcal("C:/Users/hiker/Thesis/PitmanCanyon.csv")

In [411]:
#Burro Flat Paleoseismic Site

Burro_location = 377 # km
Burro = paleoseismic_dataset("Burro", "San Andreas", Burro_location)
Burro.load_oxcal("C:/Users/hiker/Thesis/BurroFlats.csv")
Burro.event_quality.update(Burro_Event_3 = 5)

In [ ]:
#Cabezon Paleoseismic Site

Cabezon_location = #To be continued after I finish setting up the new Event Enumeration and build up the fault in CalTopo through San Gorgonio Pass

In [ ]:
##

In [1063]:
#Erik experimenting with stuff: make correlation matrix including hiatuses and basal boundaries

# Set up master event list for enumeration of candidates

included_sites = [Annette, Bidart, Frazier, Elizabeth, Pallett, Wrightwood, Pitman, Burro]
included_site_names = [site.site_name for site in included_sites]

LEFT_END = 0
LEFT_END_MINSLIP = 0
RIGHT_END = 390
RIGHT_END_MINSLIP = 0

master_event_list = [] # earthquake objects
#master_site_list = [] # paleoseismic site objects in fault-position order
master_object_list = [] # earthquake, hiatus, and basal boundary objects
master_site_list = [] # paleoseismic site objects in fault-position order

for site in included_sites:
    master_event_list = master_event_list + site.event_list()
    master_object_list = master_object_list + site.event_list() + site.hiatuses + [site.base]
    master_site_list = master_site_list + [site] * len(site.event_list()) + [site] * len(site.hiatuses) + [site] # extra + [site] at the end accounts for the basal boundary (since it has no length attribute and len(base) = 1

master_event_names = [eq.name for eq in master_event_list] # corresponding list of event names
master_site_names = [site.site_name for site in master_site_list] # corresponding list of event names
master_object_names = [obj.name for obj in master_object_list]

#master_event_names.index(Pallett.events[7].name) Example of how to find index of an event in matrix

#total_events = len(master_event_names)
total_objects = len(master_object_list)

# Construct correlation matrix
correlation_matrix = np.zeros((total_objects,total_objects))

for i in range(total_objects):
    for j in range(total_objects):
        if master_object_list[i].name.split('_', 1)[-1] == 'basal_boundary' and master_object_list[j].name.split('_', 1)[-1] == 'basal_boundary': # we have chosen to automatically correlate basal boundaries: this should not effect event enumeration in next step
            correlation_matrix[i,j] = 1        
        elif master_object_list[i].name.split('_', 1)[-1] == 'basal_boundary' or master_object_list[j].name.split('_', 1)[-1] == 'basal_boundary': # if we are correlating an event or hiatus and a basal boundary:
            idx = [master_object_list[i].name.split('_', 1)[-1], master_object_list[j].name.split('_', 1)[-1]].index('basal_boundary')
            boundary_object = [master_object_list[i], master_object_list[j]][idx] # save the '.base' object
            other_object = [master_object_list[i], master_object_list[j]][abs(idx - 1)] # save the other object
            # test if that event (or hiatus) correlate with the no - record period at that trench (basal boundary):
            if np.average(other_object.age, weights = other_object.PDF) > np.average(boundary_object.age, weights = boundary_object.PDF): # if mean(event.age) or mean(hiatus.age) younger than mean(base.age):
                # check bhattacharya coefficient of the two pdfs to decide wether to correlate
                correlation_matrix[i,j] = other_object.BC(boundary_object)
            else:
                correlation_matrix[i,j] = 1 #Set to highest possible BC, assuming that the mean of the event PDF being older than the mean of the basal boundary PDF means that the event/hiatus has to be older than the trench record
        else: #If we are only correlating events and/or hiatuses:
            correlation_matrix[i,j] = master_object_list[i].BC(master_object_list[j]) # works for hiatuses as well

In [1103]:
# Recursive candidate enumeration algorithm
candidate_events = []
RIGHT = 1
LEFT = 2
def make_candidate(correlation_row, new_object, direction = RIGHT, left_objects = [], right_objects = [], right_end = None, right_minslip = None):
    # correlation_row: one row of the correlation matrix
    # root: event at the root of this candidate tree. All events correlate to root by definition.
    # left_events: list of events in the left dirction already added to the candidate
    # right_events: list of events in the right direction already added to the candidate
    # new_event: proposed new event at the next trench site
    # direction: are we adding an event to the RIGHT or LEFT?
    
    # STEP 1: Check new event against all right and left events to make sure that this candidate is valid.
    # If not valid, then return, otherwise continue and make event and recursively call as needed.
    new_object_index = master_object_names.index(new_object.name)
    combined_objects = left_objects + right_objects
    for object in combined_objects:
        object_index = master_object_names.index(object.name)
        # Test if new event is valid correlation to each other event in the list (except root).
        # If an invalid correlation is found, we return (end the function) and don't generate a candidate 
        if correlation_matrix[new_object_index,object_index] < CORRELATION_THRESHOLD: return

    # STEP 2. Look for the next possible event (one or more) and call make_candidate for each of these. This
    # step recursively builds the candidates. To ensure against infinite recursion:
    # If the direction == RIGHT, then we look to the right and then to the left
    # If the direction == LEFT, then we only look to the left
    if direction == RIGHT:
        # find name of rightmost site
        rightmost_site_name = master_site_names[master_object_names.index(new_object.name)]
        # find the name of the next site or end of fault
        rightmost_site_index = included_site_names.index(rightmost_site_name)
        if rightmost_site_index == len(included_site_names) - 1:
            # We found the end of the fault, use this as the end point
            right_end = RIGHT_END
            right_minslip = RIGHT_END_MINSLIP
        else:
            # We look at the next trench site on the list and call next level of recursion if we find possible correlation
            next_trench_site = included_sites[rightmost_site_index + 1]
            next_trench_site_name = next_trench_site.site_name
            possible_correlative_object_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
            for object_index in possible_correlative_object_indices:
                # This is the key step in the recursion. If an event at the next trench has a possible correlation,
                # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
                # If more than one possible correlative is found make_candidate is called for each.
                if correlation_row[object_index] >= CORRELATION_THRESHOLD:
                    next_new_object = master_object_list[object_index]
                    make_candidate(correlation_row, next_new_object, RIGHT, left_objects, right_objects + [new_object])
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
            right_end = next_trench_site.position
            right_minslip = next_trench_site.slip_threshold

        # Final steps, looking to the right:
        right_objects = right_objects + [new_object] # Copy new event to right event list. List complete for this candidate.
        new_object = right_objects[0] # new_event is now the root event so we can start recursion to the left
        
    # Finished looking to right, now look left
    
    leftmost_site_name = master_site_names[master_object_names.index(new_object.name)]
    # find the name of the next site or end of fault
    leftmost_site_index = included_site_names.index(leftmost_site_name)
    if leftmost_site_index == 0:
            # We found the end of the fault, use this as the other end point
        left_end = LEFT_END
        left_minslip = LEFT_END_MINSLIP
    else:
        # We look at the next trench site on the list and call next level of recursion if we find possible correlation
        next_trench_site = included_sites[leftmost_site_index - 1]
        next_trench_site_name = next_trench_site.site_name
        possible_correlative_object_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
        for object_index in possible_correlative_object_indices:
            # This is the key step in the recursion. If an event at the next trench has a possible correlation,
            # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
            # If more than one possible correlative is foumd make_candidate is called for each.
            if correlation_row[object_index] >= CORRELATION_THRESHOLD:
                next_new_object = master_object_list[object_index]
                make_candidate(correlation_row, next_new_object, LEFT, left_objects + [new_object], right_objects, right_end, right_minslip)
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
        left_end = next_trench_site.position
        left_minslip = next_trench_site.slip_threshold
    
    # Final steps, looking to the left:
    left_objects = left_objects + [new_object] # Copy new event to left event list. List complete for this candidate.
    left_objects = left_objects[1:] # Remove the root event left_events (it is preserved in right_events)
    temporary_event_name = right_objects[0].name # name this event after the root event. Elaborate names later....

    # Final step: create candidate and add it to the list
    candidate_events.append(Candidate_Event(temporary_event_name, left_objects[::-1] + right_objects))
    candidate_events[-1].right_end = right_end
    candidate_events[-1].right_minslip = right_minslip
    candidate_events[-1].left_end = left_end
    candidate_events[-1].left_minslip = left_minslip

In [1195]:
# build master candidate event list, only using events as roots

candidate_events = []

for event in master_event_names:
    object_index = master_object_names.index(event)
    make_candidate(correlation_matrix[object_index], master_object_list[object_index])

In [1196]:
# convert candidates to matrix (IMPORTANT CHOICE: unless suggested otherwise, we will build the candidates_matrix and candidates_df using ONLY events)

candidates_matrix = np.zeros((len(candidate_events), len(master_event_list)))

for i, candidate in enumerate(candidate_events):
    for event in candidate.event_list:
        if event.name in master_event_names:
            idx = master_event_names.index(event.name)
            candidates_matrix[i][idx] = 1

# convert matrix to dataframe with extra 'event_id' column, drop duplicate candidates

candidates_df = pd.DataFrame(candidates_matrix, columns=master_event_names)
candidates_df["event_id"] = np.arange(len(candidates_df))
candidates_df.drop_duplicates(subset = master_event_names, inplace=True) # drop duplicates inplace, checking all columns except 'event_id', because 'event_id' would make each row unique
    
# remove corresponding duplicate candidates from candidate_events based on 'event_id'

candidate_events = [item for i, item in enumerate(candidate_events) if i in candidates_df['event_id']]

377

In [ ]:
#Paste in cells from thesis condensed and try to get all cells to work

In [ ]:
#Make duplicate of notebook and synthesize with 'Thesis Condensed to Coachella' to get new results through San Gorgonio Pass to Indio Stran

In [ ]:
##

In [725]:
# Set up master event list for enumeration of candidates

included_sites = [Annette, Frazier, Pallett]
included_site_names = [site.site_name for site in included_sites]

LEFT_END = 0
LEFT_END_MINSLIP = 0
RIGHT_END = 390
RIGHT_END_MINSLIP = 0

master_event_list = [] # earthquake objects
master_site_list = [] # paleoseismic site objects in fault-position order

for site in included_sites:
    master_event_list = master_event_list + site.event_list()
    master_site_list = master_site_list + [site] * len(site.event_list())

master_event_names = [eq.name for eq in master_event_list] # corresponding list of event names
master_site_names = [site.site_name for site in master_site_list] # corresponding list of event names

#master_event_names.index(Pallett.events[7].name) Example of how to find index of an event in matrix

total_events = len(master_event_names)

# Construct correlation matrix
correlation_matrix = np.zeros((total_events,total_events))

for i in range(total_events):
    for j in range(total_events):
        correlation_matrix[i,j] = master_event_list[i].BC(master_event_list[j])

# Recursive candidate enumeration algorithm
candidate_events = []
RIGHT = 1
LEFT = 2
def make_candidate(correlation_row, new_event, direction = RIGHT, left_events = [], right_events=[], right_end = None, right_minslip = None):
    # correlation_row: one row of the correlation matrix
    # root: event at the root of this candidate tree. All events correlate to root by definition.
    # left_events: list of events in the left dirction already added to the candidate
    # right_events: list of events in the right direction already added to the candidate
    # new_event: proposed new event at the next trench site
    # direction: are we adding an event to the RIGHT or LEFT?
    
    # STEP 1: Check new event against all right and left events to make sure that this candidate is valid.
    # If not valid, then return, otherwise continue and make event and recursively call as needed.
    new_event_index = master_event_names.index(new_event.name)
    combined_events = left_events + right_events
    for event in combined_events:
        event_index = master_event_names.index(event.name)
        # Test if new event is valid correlation to each other event in the list (except root).
        # If an invalid correlation is found, we return (end the function) and don't generate a candidate 
        if correlation_matrix[new_event_index,event_index] < CORRELATION_THRESHOLD: return

    # STEP 2. Look for the next possible event (one or more) and call make_candidate for each of these. This
    # step recursively builds the candidates. To ensure against infinite recursion:
    # If the direction == RIGHT, then we look to the right and then to the left
    # If the direction == LEFT, then we only look to the left
    if direction == RIGHT:
        # find name of rightmost site name
        rightmost_site_name = master_site_names[master_event_names.index(new_event.name)]
        # find the name of the next site or end of fault
        rightmost_site_index = included_site_names.index(rightmost_site_name)
        if rightmost_site_index == len(included_site_names) - 1:
            # We found the end of the fault, use this as the end point
            right_end = RIGHT_END
            right_minslip = RIGHT_END_MINSLIP
        else:
            # We look at the next trench site on the list and call next level of recursion if we find possible correlation
            next_trench_site = included_sites[rightmost_site_index + 1]
            next_trench_site_name = next_trench_site.site_name
            possible_correlative_event_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
            for event_index in possible_correlative_event_indices:
                # This is the key step in the recursion. If an event at the next trench has a possible correlation,
                # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
                # If more than one possible correlative is foumd make_candidate is called for each.
                if correlation_row[event_index] >= CORRELATION_THRESHOLD:
                    next_new_event = master_event_list[event_index]
                    make_candidate(correlation_row, next_new_event, RIGHT, left_events, right_events + [new_event])
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
            right_end = next_trench_site.position
            right_minslip = next_trench_site.slip_threshold

        # Final steps, looking to the right:
        right_events = right_events + [new_event] # Copy new event to right event list. List complete for this candidate.
        new_event = right_events[0] # new_event is now the root event so we can start recursion to the left
        
    # Finished looking to right, now look left
    
    leftmost_site_name = master_site_names[master_event_names.index(new_event.name)]
    # find the name of the next site or end of fault
    leftmost_site_index = included_site_names.index(leftmost_site_name)
    if leftmost_site_index == 0:
            # We found the end of the fault, use this as the other end point
        left_end = LEFT_END
        left_minslip = LEFT_END_MINSLIP
    else:
        # We look at the next trench site on the list and call next level of recursion if we find possible correlation
        next_trench_site = included_sites[leftmost_site_index - 1]
        next_trench_site_name = next_trench_site.site_name
        possible_correlative_event_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
        for event_index in possible_correlative_event_indices:
            # This is the key step in the recursion. If an event at the next trench has a possible correlation,
            # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
            # If more than one possible correlative is foumd make_candidate is called for each.
            if correlation_row[event_index] >= CORRELATION_THRESHOLD:
                next_new_event = master_event_list[event_index]
                make_candidate(correlation_row, next_new_event, LEFT, left_events + [new_event], right_events, right_end, right_minslip)
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
        left_end = next_trench_site.position
        left_minslip = next_trench_site.slip_threshold
    
    # Final steps, looking to the left:
    left_events = left_events + [new_event] # Copy new event to left event list. List complete for this candidate.
    left_events = left_events[1:] # Remove the root event left_events (it is preserced in right_events)
    temporary_event_name = right_events[0].name # name this event after the root event. Elaborate names later....

    # Final step: create candidate and add it to the list
    candidate_events.append(Candidate_Event(temporary_event_name, left_events[::-1] + right_events))
    candidate_events[-1].right_end = right_end
    candidate_events[-1].right_minslip = right_minslip
    candidate_events[-1].left_end = left_end
    candidate_events[-1].left_minslip = left_minslip
    

    



In [743]:
candidate_events = []
start_event = 8
make_candidate(correlation_matrix[start_event,:],master_event_list[start_event])
#correlation_matrix[6,:]

IndexError: list index out of range

In [729]:
for x in range(len(candidate_events)):
    print([event.name for event in candidate_events[x].event_list])
#indices = [i for i, x in enumerate(master_site_names) if x == 'Pallett']
#indices

In [81]:
for x in range(len(candidate_events)):
    print(candidate_events[x].left_end)

0
20
0
20
0
20


In [ ]:
## START OF ERIK EXPERIMENTING WITH STUFF

In [ ]:
#Edit preexisting master_event_list and event enumeration code to work with hiatuses

#Fundamental difference is that the single correlation matrix for events and hiatuses will be built with two different correlation techniques:
#event-to-event correlation will be decided based on Bhattacharya Coefficient (with the idea that different PDFs that represent the same event should
#overlap well, meaning their BC will be relatively high), whereas event-to-hiatus and hiatus-to-hiatus correlation will be conducted based on the overlap
#of their 95th percentile age range (this will work as long as we are still testing every possible pair in the string of pearls for correlation)
#(with the idea that a hiatus overlapping with an event or a hiatus overlapping with a hiatus does not imply anything about the likelihood of the event
#being correlated with the hiatus, only stating that it is possible that that hiatus concealed the given event)

In [ ]:
#Create correlation matrix to hold both events and hiatuses: instead of a function like .BC outputting a single number based
#on PDF agreement, the .CDF_overlap function will output only a 'TRUE' or 'FALSE', based on whether their x percentile ages overlap

